In [ ]:
# Please edit the path according to your PC environment
from google.colab import drive
drive.mount('/content/drive')

!ls '/content/drive/MyDrive/counterfactual_pitch_sequence_analysis/'
%cd "/content/drive/MyDrive/counterfactual_pitch_sequence_analysis/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
data  program
/content/drive/MyDrive/counter_fluctucal_analysis


In [ ]:
# Install the pybaseball library to handle Statcast data.
!pip install pybaseball

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.1/426.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.7/449.7 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 13.0 MB/s eta 0:00:00


In [ ]:
from pathlib import Path
import re
import time
import pandas as pd
from pybaseball import statcast_pitcher, playerid_lookup, cache

cache.enable()

input_dir = Path("./data/stats/pitcher")
output_base_dir = Path("./data/pitch_sequence")

# Exclude the shortened 2020 season
years = [year for year in range(2017, 2026) if year != 2020]

# Define helper functions
def split_player_name(full_name):
    parts = str(full_name).strip().split()
    first_name = parts[0]
    last_name = parts[-1]
    return first_name, last_name

def find_mlbam_id_from_name(full_name, year):
    first_name, last_name = split_player_name(full_name)
    lookup = playerid_lookup(last_name, first_name)
    if "mlb_played_first" in lookup.columns and "mlb_played_last" in lookup.columns:
        lookup_year = lookup[
            (lookup["mlb_played_first"].fillna(9999) <= year) &
            (lookup["mlb_played_last"].fillna(0) >= year)
        ]
        if not lookup_year.empty:
            lookup = lookup_year
    player_id = int(lookup.iloc[0]["key_mlbam"])
    return player_id

def get_pitcher_id(row, year):
    id_columns = ["player_id", "key_mlbam", "mlbam_id", "MLBAMID"]
    for col in id_columns:
        if col in row.index and pd.notna(row[col]):
            return int(row[col])
    return find_mlbam_id_from_name(row["Name"], year)

def download_pitcher_statcast(name, player_id, year, output_path):
    start_dt = f"{year}-01-01"
    end_dt = f"{year}-12-31"
    df = statcast_pitcher(start_dt, end_dt, player_id)
    if not df.empty:
        df.insert(0, "Season", year)
        df.insert(1, "Name", name)
        df.insert(2, "player_id", player_id)
    df.to_csv(output_path, index=False, encoding="utf-8-sig")
    return len(df)

In [ ]:
# Download target pitchers' data for each year
for year in years:
    input_path = input_dir / f"{year}.xlsx"
    output_dir = output_base_dir / str(year)
    output_dir.mkdir(parents=True, exist_ok=True)

    pitchers = pd.read_excel(input_path)

    print(f"\n===== {year} year: {len(pitchers)} pitchers =====")
    for _, row in pitchers.iterrows():
        name = "_".join(str(row["Name"]).strip().split())
        filename = name + ".csv"
        output_path = output_dir / filename

        player_id = get_pitcher_id(row, year)

        n_pitches = download_pitcher_statcast(
            name=name,
            player_id=player_id,
            year=year,
            output_path=output_path
        )

        print(f"[OK] {year} {name}: {n_pitches} pitches -> {output_path}")

        time.sleep(1)


===== 2017 year: 58 pitchers =====
[OK] 2017 Corey_Kluber: 3312 pitches -> data/pitch_sequence/2017/Corey_Kluber.csv
[OK] 2017 Clayton_Kershaw: 3302 pitches -> data/pitch_sequence/2017/Clayton_Kershaw.csv
[OK] 2017 Max_Scherzer: 3354 pitches -> data/pitch_sequence/2017/Max_Scherzer.csv
[OK] 2017 Stephen_Strasburg: 3062 pitches -> data/pitch_sequence/2017/Stephen_Strasburg.csv
[OK] 2017 Robbie_Ray: 3238 pitches -> data/pitch_sequence/2017/Robbie_Ray.csv
[OK] 2017 Chris_Sale: 3756 pitches -> data/pitch_sequence/2017/Chris_Sale.csv
[OK] 2017 Gio_Gonzalez: 3676 pitches -> data/pitch_sequence/2017/Gio_Gonzalez.csv
[OK] 2017 Luis_Severino: 3509 pitches -> data/pitch_sequence/2017/Luis_Severino.csv
[OK] 2017 Marcus_Stroman: 3198 pitches -> data/pitch_sequence/2017/Marcus_Stroman.csv
[OK] 2017 Zack_Greinke: 3499 pitches -> data/pitch_sequence/2017/Zack_Greinke.csv
[OK] 2017 Ervin_Santana: 3410 pitches -> data/pitch_sequence/2017/Ervin_Santana.csv
[OK] 2017 Carlos_Carrasco: 3297 pitches -> dat

In [ ]:
from google.colab import runtime
runtime.unassign()